In [1]:
import pandas as pd

decisao = pd.read_csv("../data/eventos_com_decisao.csv")
spy = pd.read_csv("../data/spy_precos.csv")

decisao = decisao.merge(spy[["data", "retorno_pct"]], on="data", how="left")

operacoes = decisao[decisao["opera"]].copy()
print(operacoes[["data", "direcao", "tamanho_posicao", "retorno_pct"]])

          data  direcao  tamanho_posicao  retorno_pct
1   2023-04-12  comprar         0.422230    -0.407599
13  2024-04-10   vender         0.281011    -1.001323
33  2026-02-13  comprar         0.365396     0.070450
35  2026-04-10  comprar         0.293775    -0.066178
38  2026-07-14  comprar         0.293153     0.355064


In [2]:
def calcular_retorno_operacao(row):
    if row["direcao"] == "comprar":
        return row["retorno_pct"] * row["tamanho_posicao"]
    elif row["direcao"] == "vender":
        return -row["retorno_pct"] * row["tamanho_posicao"]
    return 0

operacoes["retorno_operacao"] = operacoes.apply(calcular_retorno_operacao, axis=1)
print(operacoes[["data", "direcao", "retorno_pct", "tamanho_posicao", "retorno_operacao"]])

          data  direcao  retorno_pct  tamanho_posicao  retorno_operacao
1   2023-04-12  comprar    -0.407599         0.422230         -0.172101
13  2024-04-10   vender    -1.001323         0.281011          0.281383
33  2026-02-13  comprar     0.070450         0.365396          0.025742
35  2026-04-10  comprar    -0.066178         0.293775         -0.019441
38  2026-07-14  comprar     0.355064         0.293153          0.104088


In [3]:
capital_inicial = 100
operacoes = operacoes.sort_values("data")
operacoes["capital"] = capital_inicial * (1 + operacoes["retorno_operacao"] / 100).cumprod()

print(operacoes[["data", "retorno_operacao", "capital"]])

          data  retorno_operacao     capital
1   2023-04-12         -0.172101   99.827899
13  2024-04-10          0.281383  100.108798
33  2026-02-13          0.025742  100.134568
35  2026-04-10         -0.019441  100.115101
38  2026-07-14          0.104088  100.219309


In [4]:
retorno_total = (operacoes["capital"].iloc[-1] / capital_inicial - 1) * 100
volatilidade = operacoes["retorno_operacao"].std()
sharpe = operacoes["retorno_operacao"].mean() / operacoes["retorno_operacao"].std()
win_rate = (operacoes["retorno_operacao"] > 0).mean() * 100

pico = operacoes["capital"].cummax()
drawdown = (operacoes["capital"] - pico) / pico
max_drawdown = drawdown.min() * 100

print(f"Retorno total: {retorno_total:.2f}%")
print(f"Volatilidade (por operação): {volatilidade:.2f}%")
print(f"Índice de Sharpe (não anualizado): {sharpe:.2f}")
print(f"Máx. drawdown: {max_drawdown:.2f}%")
print(f"Win rate: {win_rate:.2f}%")

Retorno total: 0.22%
Volatilidade (por operação): 0.17%
Índice de Sharpe (não anualizado): 0.26
Máx. drawdown: -0.02%
Win rate: 60.00%


In [5]:
data_inicio = operacoes["data"].min()
data_fim = operacoes["data"].max()

spy_periodo = spy[(spy["data"] >= data_inicio) & (spy["data"] <= data_fim)]
preco_inicial = spy_periodo["close"].iloc[0]
preco_final = spy_periodo["close"].iloc[-1]

retorno_benchmark = (preco_final / preco_inicial - 1) * 100
print(f"Retorno do benchmark (SPY buy-and-hold): {retorno_benchmark:.2f}%")

Retorno do benchmark (SPY buy-and-hold): 91.96%


In [6]:
tabela_resultado = pd.DataFrame({
    "Métrica": ["Retorno total (%)", "Volatilidade (%)", "Índice de Sharpe", "Máx. drawdown (%)", "Win rate (%)"],
    "FARO": [round(retorno_total, 2), round(volatilidade, 2), round(sharpe, 2), round(max_drawdown, 2), round(win_rate, 2)],
    "Benchmark": [round(retorno_benchmark, 2), "-", "-", "-", "-"]
})
print(tabela_resultado)

             Métrica   FARO Benchmark
0  Retorno total (%)   0.22     91.96
1   Volatilidade (%)   0.17         -
2   Índice de Sharpe   0.26         -
3  Máx. drawdown (%)  -0.02         -
4       Win rate (%)  60.00         -


In [1]:
import pandas as pd

decisao = pd.read_csv("../data/eventos_com_decisao.csv")
spy = pd.read_csv("../data/spy_precos.csv")
ewz = pd.read_csv("../data/ewz_precos.csv")

spy["ativo"] = "SPY"
ewz["ativo"] = "EWZ"
precos_todos = pd.concat([spy, ewz], ignore_index=True)

playbooks = pd.read_csv("../data/playbooks.csv")
decisao = decisao.merge(playbooks[["indicador", "ativo_alvo"]], on="indicador", how="left")

decisao = decisao.merge(
    precos_todos[["data", "ativo", "retorno_pct"]],
    left_on=["data", "ativo_alvo"],
    right_on=["data", "ativo"],
    how="left"
)

operacoes = decisao[decisao["opera"]].copy()
print(operacoes[["data", "indicador", "ativo_alvo", "direcao", "retorno_pct"]])

           data indicador ativo_alvo  direcao  retorno_pct
2    2023-04-12   CPI_EUA        SPY  comprar    -0.407599
3    2023-04-12   CPI_EUA        SPY  comprar    -0.407599
26   2024-04-10   CPI_EUA        SPY   vender    -1.001323
27   2024-04-10   CPI_EUA        SPY   vender    -1.001323
66   2026-02-13   CPI_EUA        SPY  comprar     0.070450
67   2026-02-13   CPI_EUA        SPY  comprar     0.070450
70   2026-04-10   CPI_EUA        SPY  comprar    -0.066178
71   2026-04-10   CPI_EUA        SPY  comprar    -0.066178
76   2026-07-14   CPI_EUA        SPY  comprar     0.355064
77   2026-07-14   CPI_EUA        SPY  comprar     0.355064
91   2024-02-01   IPCA_BR        EWZ   vender     1.123939
102  2025-01-01   IPCA_BR        EWZ  comprar          NaN
103  2025-02-01   IPCA_BR        EWZ   vender          NaN
104  2025-03-01   IPCA_BR        EWZ   vender          NaN
